# Phase 6b Stage 2 — train the two ablation finalists (Kaggle)

Trains `w125` (P3/P4/P5, width 0.125) and `p34` (P3/P4, width 0.25) from random init and compares accuracy — the piece Phase 6a deliberately left undone (it measured flash/RAM cost only, no training).

**Moved here from Colab**: free-tier Colab GPU quota couldn't absorb two 40-epoch runs plus the ~874MB dataset transfer in one sitting. Kaggle's free tier gives 30 GPU-hours/week (vs Colab's rolling, less predictable cap) and — more importantly — the ShanghaiTech dataset is already hosted on Kaggle, so it attaches to the notebook directly via **Add Data** instead of downloading through the Kaggle API. No `kaggle.json` upload needed here.

**Epoch budget cut to 20** (from 40) with `--patience 8` (40% of the cap, matching the ratio used for the killed local run's 15/40). Both variants trained comfortably past their early gains within the first ~10-15 epochs in the partial local run, so 20 epochs + early stopping should still land near the accuracy plateau without burning the whole weekly quota on one experiment.

**Caveat carried over from the local/Colab attempts**: `--cache` stays off. This dataset decodes to ~19GB uncached; Kaggle's free GPU notebooks give ~13GB system RAM, so caching to RAM risks the same kind of exhaustion that hung the local M4 Pro (unified memory there; here it would just be an OOM-killed kernel instead of a hung machine, but still a failure to avoid). Plain disk reads are fine on a real GPU.

**Before running:**
1. Notebook settings (right sidebar) -> **Accelerator** -> GPU T4 x2 or P100.
2. **Internet** -> On (needed for `git clone` and `pip install`).
3. **Add Data** -> search `nikanvasei/shanghaitech-campus-dataset-test` -> Add. It will mount read-only under `/kaggle/input/`.

Then run cells top to bottom.

## 1. Clone the repo and install dependencies

In [ ]:
%cd /kaggle/working
!git clone -b phase/six https://github.com/akash-reddy-k/edgeAi.git
%cd edgeAi
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator > GPU before continuing")

## 2. Locate the attached dataset

No download needed — Kaggle mounted it read-only under `/kaggle/input/` when you added it via **Add Data**. This cell finds `frames/` and `SHANGHAI_test.txt` wherever the dataset owner's zip structure put them (same defensive search the Colab notebook uses for the API download, since Kaggle "Add Data" and `kaggle datasets download --unzip` can both nest one level deep) and **copies** — not moves, `/kaggle/input` is read-only — them into the layout `prepare_dataset.py` expects.

In [ ]:
import shutil
from pathlib import Path

base = Path("data/SHANGHAI_Test")
base.mkdir(parents=True, exist_ok=True)
kaggle_input = Path("/kaggle/input")

assert kaggle_input.exists() and any(kaggle_input.iterdir()), (
    "Nothing under /kaggle/input — add the dataset via the Add Data panel "
    "(search nikanvasei/shanghaitech-campus-dataset-test) before running this cell."
)

if not (base / "frames").exists():
    candidates = [p for p in kaggle_input.rglob("frames") if p.is_dir()]
    assert candidates, "No frames/ directory found under /kaggle/input — inspect the attached dataset manually"
    shutil.copytree(candidates[0], base / "frames")

if not (base / "SHANGHAI_test.txt").exists():
    candidates = list(kaggle_input.rglob("SHANGHAI_test.txt"))
    assert candidates, "No SHANGHAI_test.txt found under /kaggle/input — inspect the attached dataset manually"
    shutil.copy(candidates[0], base / "SHANGHAI_test.txt")

n_scenes = len(list((base / "frames").iterdir()))
print(f"OK — {n_scenes} scenes under {base}/frames")

## 3. Base weights + pseudo-labeled dataset

`prepare_dataset.py` uses stock `yolov8n.pt` as a pseudo-labeler (COCO -> campus domain adaptation), same as Phase 3 locally.

In [ ]:
import os, shutil
from ultralytics import YOLO

os.makedirs("models", exist_ok=True)
YOLO("yolov8n.pt")  # downloads to cwd if not already cached
if os.path.exists("yolov8n.pt") and not os.path.exists("models/yolov8n.pt"):
    shutil.move("yolov8n.pt", "models/yolov8n.pt")
print("models/yolov8n.pt present:", os.path.exists("models/yolov8n.pt"))

In [ ]:
!python prepare_dataset.py

## 4. Train the two Stage 2 finalists

Both from random init (`--variant` implies `pretrained=False` in `train.py` — see its docstring for why). `--device 0` targets the Kaggle GPU. No `--cache`: kept off deliberately, see the caveat at the top. `--epochs 20 --patience 8`: cut from the original 40/15 budget to fit Kaggle's weekly GPU quota across both runs.

In [ ]:
!python train.py --variant w125 --epochs 20 --patience 8 --batch 16 --device 0

In [ ]:
!python train.py --variant p34 --epochs 20 --patience 8 --batch 16 --device 0

## 5. Package results for download

Zips both run directories (`results.csv`, `args.yaml`, plots) and the best weights into `/kaggle/working/stage2_results.zip`. Kaggle persists everything under `/kaggle/working/` as the notebook's **Output** — after the run finishes (or you Save Version), download it from the Output tab in the right sidebar (no `files.download()` here; that's a Colab-only API). Unzip locally into the same paths — `training_runs/*` -> `runs/detect/results/training/`, `models/*.pt` -> `models/` — and everything downstream (the Stage 2 comparison, README) reads it exactly like a local run.

In [ ]:
import shutil, os

stage_dir = "/kaggle/working/stage2_package"
os.makedirs(f"{stage_dir}/models", exist_ok=True)
shutil.copytree("runs/detect/results/training", f"{stage_dir}/training_runs", dirs_exist_ok=True)

for f in ["models/yolov8_ablation_w125.pt", "models/yolov8_ablation_p34.pt"]:
    if os.path.exists(f):
        shutil.copy(f, f"{stage_dir}/models/")
    else:
        print(f"WARNING: {f} missing — training may not have completed")

shutil.make_archive("/kaggle/working/stage2_results", "zip", stage_dir)
print("Wrote /kaggle/working/stage2_results.zip — download it from the Output tab.")

In [ ]:
# Quick summary printed here too, in case you'd rather not download and unzip —
# paste this back into the conversation.
import pandas as pd

for variant in ["w125", "p34"]:
    csv_path = f"runs/detect/results/training/yolov8_ablation_{variant}/results.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        print(f"--- {variant} ---")
        print(f"  epochs run : {len(df)}")
        print(f"  mAP50      : {last['metrics/mAP50(B)']:.4f}")
        print(f"  mAP50-95   : {last['metrics/mAP50-95(B)']:.4f}")
        print(f"  Precision  : {last['metrics/precision(B)']:.4f}")
        print(f"  Recall     : {last['metrics/recall(B)']:.4f}\n")